# Build & seed multi-reference examples into the Playground

For each example dataset we build an **Icechunk** store on **two** hosts — **OSN**
and **AWS S3** — and seed the dataset's STAC Item into the local ESGF-Playground
with **multiple, separate virtual-reference assets**:

| asset key | engine | storage | source data |
|---|---|---|---|
| `reference_file` | kerchunk | CEDA HTTP | (existing, CEDA only) |
| `reference_icechunk_osn` | icechunk | OSN | CEDA HTTP / esgf-world S3 |
| `reference_icechunk_s3`  | icechunk | AWS S3 | CEDA HTTP / esgf-world S3 |

These are **separate assets, not `alternate`s**: the alternate-assets extension is
only for *identical files* (same checksum/size). Distinct virtual stores fail that
test. One example is sourced from **esgf-world S3** so its virtual chunks live on
S3 — the only case where xpystac's S3-only read can succeed (see the discovery
notebook).

**Prereqs:** Playground up (`docker compose up -d` in `~/Code/ESGF-Playground`);
AWS SSO logged in; OSN keys in 1Password. Fill in the `S3_*` / `AWS_PROFILE` config.

In [ ]:
import os
import subprocess

import httpx
import icechunk as ic
import obstore
from obstore.store import from_url

from cmip7_virtualization import (
    aws_s3_storage,
    build_reference_asset,
    osn_storage,
    repo_exists,
    urls_from_stac_item,
    vccs_from_registry,
    virtualize_from_urls,
)

In [ ]:
# --- catalog / collection ---
LOCAL_STAC = "http://localhost:9010"        # Playground East (stac-fastapi-es)
CEDA_STAC  = "https://api.stac.esgf.ceda.ac.uk"
COLLECTION = "CMIP6"
N_CEDA_ITEMS = 2

# --- OSN (existing public-read bucket; write keys via 1Password) ---
OSN_BUCKET = "leap-pangeo-pipeline"
OSN_ENDPOINT = "https://nyu1.osn.mghpcc.org"
OSN_PREFIX_ROOT = "cmip7-virtualization"
def op_read(ref): return subprocess.check_output(["op", "read", ref]).decode().strip()
OSN_KEY    = op_read("op://Work/z6baienaiyhiexztlbbonbeaka/Read-Write/Access_Key")
OSN_SECRET = op_read("op://Work/z6baienaiyhiexztlbbonbeaka/Read-Write/Secret_Access_Key")

# --- AWS S3 (carbonplan-cmip7 account; auth via the AWS default chain / SSO) ---
os.environ.setdefault("AWS_PROFILE", "carbonplan-cmip7")
S3_BUCKET = "carbonplan-cmip7"
S3_REGION = "us-east-1"
S3_PREFIX_ROOT = "cmip7-virtualization"

# esgf-world (public CMIP6 S3 mirror) is in us-east-2
ESGF_WORLD_REGION = "us-east-2"

In [ ]:
# Playground health check
r = httpx.get(LOCAL_STAC, timeout=5); r.raise_for_status()
print("Playground up:", r.json().get("title"))

## Gather example datasets (CEDA HTTP-sourced + one esgf-world S3-sourced)

In [ ]:
# CEDA items (HTTP source)
items = httpx.get(f"{CEDA_STAC}/collections/{COLLECTION}/items?limit=20", timeout=30).json()["features"]
ceda_items = [i for i in items if urls_from_stac_item(i)][:N_CEDA_ITEMS]
print("CEDA items:", [i["id"] for i in ceda_items])

# esgf-world item (S3 source, anonymous, us-east-2)
ESGFWORLD_ID = "CMIP6.CMIP.CCCma.CanESM5.historical.r10i1p1f1.Omon.uo.gn.v20190429"
_prefix = "CMIP6/CMIP/CCCma/CanESM5/historical/r10i1p1f1/Omon/uo/gn/v20190429/"
_st = from_url("s3://esgf-world", skip_signature=True, region=ESGF_WORLD_REGION)
esgfworld_urls = [
    f"s3://esgf-world/{o['path']}"
    for o in obstore.list_with_delimiter(_st, prefix=_prefix)["objects"]
    if o["path"].endswith(".nc")
]
print("esgf-world files:", len(esgfworld_urls))

# (id, urls, source_node, source_kind)
datasets = [(i["id"], urls_from_stac_item(i), "ceda", "http") for i in ceda_items]
datasets.append((ESGFWORLD_ID, esgfworld_urls, "esgf-world", "s3"))

## Build Icechunk stores on OSN **and** AWS S3

In [ ]:
def build_store(urls, storage):
    if repo_exists(storage):
        print("    exists, skipping"); return
    vds, registry = virtualize_from_urls(urls, s3_region=ESGF_WORLD_REGION)
    config = ic.RepositoryConfig.default()
    for vcc in vccs_from_registry(registry, s3_region=ESGF_WORLD_REGION):
        config.set_virtual_chunk_container(vcc)
    repo = ic.Repository.open_or_create(storage=storage, config=config)
    session = repo.writable_session("main")
    vds.vz.to_icechunk(session.store)
    session.commit("multi-ref build")
    repo.save_config()
    print("    built")


built = {}
for cmip_id, urls, source_node, source_kind in datasets:
    print(cmip_id)
    osn_prefix = f"{OSN_PREFIX_ROOT}/{cmip_id}/"
    s3_prefix  = f"{S3_PREFIX_ROOT}/{cmip_id}/"
    print("  OSN:")
    build_store(urls, osn_storage(OSN_BUCKET, osn_prefix, OSN_KEY, OSN_SECRET))
    print("  S3:")
    build_store(urls, aws_s3_storage(S3_BUCKET, s3_prefix, S3_REGION))
    built[cmip_id] = dict(
        source_node=source_node, source_kind=source_kind, urls=urls,
        osn_href=f"s3://{OSN_BUCKET}/{osn_prefix}",
        s3_href=f"s3://{S3_BUCKET}/{s3_prefix}",
    )

## Seed the Playground with multi-reference Items (separate assets, direct PUT)

In [ ]:
# mirror collection
coll = httpx.get(f"{CEDA_STAC}/collections/{COLLECTION}", timeout=30).json()
for f in ("assets", "links"): coll.pop(f, None)
rc = httpx.post(f"{LOCAL_STAC}/collections", json=coll, timeout=30)
print("collection:", rc.status_code)


def minimal_item(cmip_id, urls):
    return {
        "type": "Feature", "stac_version": "1.1.0", "id": cmip_id, "collection": COLLECTION,
        "geometry": {"type": "Polygon", "coordinates": [[[-180, -90], [180, -90], [180, 90], [-180, 90], [-180, -90]]]},
        "bbox": [-180, -90, 180, 90],
        "properties": {"datetime": None, "start_datetime": "1850-01-01T00:00:00Z",
                       "end_datetime": "2014-12-31T00:00:00Z", "retracted": False},
        "assets": {f"data{i:04d}": {"href": u, "type": "application/netcdf", "roles": ["data"]}
                   for i, u in enumerate(urls)},
    }


ceda_by_id = {i["id"]: i for i in ceda_items}
for cmip_id, info in built.items():
    item = ceda_by_id.get(cmip_id) or minimal_item(cmip_id, info["urls"])
    a = item.setdefault("assets", {})
    a["reference_icechunk_osn"] = build_reference_asset(
        "icechunk", "osn", info["osn_href"], source_node=info["source_node"],
        region="us-east-1", anonymous=True, endpoint_url=OSN_ENDPOINT)
    a["reference_icechunk_s3"] = build_reference_asset(
        "icechunk", "s3", info["s3_href"], source_node=info["source_node"], region=S3_REGION)
    item.pop("links", None)
    r = httpx.put(f"{LOCAL_STAC}/collections/{COLLECTION}/items/{cmip_id}", json=item, timeout=30)
    print(cmip_id, "->", r.status_code)
    r.raise_for_status()

## Verify — each Item now has multiple reference assets

In [ ]:
for cmip_id in built:
    it = httpx.get(f"{LOCAL_STAC}/collections/{COLLECTION}/items/{cmip_id}", timeout=30).json()
    print(cmip_id)
    for k, asset in it["assets"].items():
        if "reference" in k or "virtual" in asset.get("roles", []):
            print("   ", k, "->", asset.get("type"), asset.get("cmip7:storage", ""))